In [ ]:

# TASK 6: NETFLIX CONTENT ANALYTICS & INSIGHTS ENGINE


import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import MultiLabelBinarizer, OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    StratifiedKFold,
    cross_val_score
)
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    ConfusionMatrixDisplay
)



DATA_CANDIDATES = [
    "data/Dataset.csv",
    "Dataset.csv",
    "Dataset(1).csv",
]
DATA_PATH = next((path for path in DATA_CANDIDATES if os.path.exists(path)), None)

if DATA_PATH is None:
    raise FileNotFoundError(
        "Dataset not found. Place it at data/Dataset.csv or next to this script."
    )

RANDOM_STATE = 42
os.makedirs("results", exist_ok=True)
os.makedirs("figures", exist_ok=True)

GROUP_ORDER = ["Kids", "Family", "Teen", "Adult"]

print("=" * 65)
print("NETFLIX CONTENT ANALYTICS & INSIGHTS ENGINE")
print("Main model candidate: Extra Trees Classifier")
print("=" * 65)



# 1. LOAD AND CLEAN DATA


df = pd.read_csv(DATA_PATH)

required_columns = [
    "title", "type", "country", "date_added",
    "release_year", "rating", "listed_in"
]
missing = [column for column in required_columns if column not in df.columns]
if missing:
    raise ValueError(f"Dataset is missing columns: {missing}")

df = df.drop_duplicates(
    subset=["title", "type", "release_year"]
).reset_index(drop=True)

for column in ["title", "type", "country", "rating", "listed_in"]:
    df[column] = (
        df[column].astype("string").str.strip()
        .replace({
            "": pd.NA,
            "Not Given": pd.NA,
            "nan": pd.NA,
            "None": pd.NA
        })
    )

df["release_year"] = pd.to_numeric(df["release_year"], errors="coerce")
df["date_added_parsed"] = pd.to_datetime(df["date_added"], errors="coerce")
df["year_added"] = df["date_added_parsed"].dt.year

df["main_country"] = (
    df["country"].fillna("Unknown")
    .str.split(",").str[0].str.strip()
    .replace("", "Unknown").fillna("Unknown")
)
df["type"] = df["type"].fillna("Unknown")
df["listed_in"] = df["listed_in"].fillna("Unknown")
df["rating"] = df["rating"].fillna("Unrated")

print("\nDataset shape:", df.shape)
print("\nContent type distribution:")
print(df["type"].value_counts())
print("\nMissing values:")
print(df.isna().sum())

df.to_csv("results/task6_cleaned_dataset.csv", index=False)



# 2. EXPLORATORY DATA ANALYSIS

df["type"].value_counts().plot(kind="bar", figsize=(7, 4))
plt.title("Netflix Content Type Distribution")
plt.xlabel("Content type")
plt.ylabel("Number of titles")
plt.tight_layout()
plt.savefig("figures/task6_content_type.png", dpi=150)
plt.close()

genre_counts = (
    df["listed_in"].dropna().str.split(",").explode().str.strip()
)
genre_counts = genre_counts[
    genre_counts.ne("Unknown") & genre_counts.ne("")
]
top_genres = genre_counts.value_counts().head(12)
top_genres.sort_values().plot(kind="barh", figsize=(9, 6))
plt.title("Top 12 Netflix Genres")
plt.xlabel("Number of genre occurrences")
plt.tight_layout()
plt.savefig("figures/task6_top_genres.png", dpi=150)
plt.close()

top_countries = (
    df.loc[df["main_country"] != "Unknown", "main_country"]
    .value_counts().head(10)
)
top_countries.sort_values().plot(kind="barh", figsize=(9, 5))
plt.title("Top 10 Countries by Listed Titles")
plt.xlabel("Number of titles")
plt.tight_layout()
plt.savefig("figures/task6_top_countries.png", dpi=150)
plt.close()

release_counts = (
    df["release_year"].dropna().astype(int).value_counts().sort_index()
)
release_counts = release_counts[
    (release_counts.index >= 1990) & (release_counts.index <= 2025)
]
release_counts.plot(figsize=(10, 5))
plt.title("Netflix Titles by Release Year")
plt.xlabel("Release year")
plt.ylabel("Number of titles in dataset")
plt.tight_layout()
plt.savefig("figures/task6_release_year_trend.png", dpi=150)
plt.close()

added_counts = (
    df["year_added"].dropna().astype(int).value_counts().sort_index()
)
added_counts.plot(kind="bar", figsize=(10, 4))
plt.title("Netflix Titles by Year Added")
plt.xlabel("Year added")
plt.ylabel("Number of titles")
plt.tight_layout()
plt.savefig("figures/task6_year_added.png", dpi=150)
plt.close()



# 3. FEATURE ENGINEERING

def split_genres(value):
    if pd.isna(value):
        return []
    return [
        item.strip()
        for item in str(value).split(",")
        if item.strip() and item.strip() != "Unknown"
    ]


class GenreEncoder(BaseEstimator, TransformerMixin):
    """Multi-hot encode genre lists while learning categories during fit."""

    def fit(self, X, y=None):
        values = X.iloc[:, 0] if isinstance(X, pd.DataFrame) else pd.Series(
            np.asarray(X).reshape(-1)
        )
        self.encoder_ = MultiLabelBinarizer()
        self.encoder_.fit(values.apply(split_genres))
        return self

    def transform(self, X):
        values = X.iloc[:, 0] if isinstance(X, pd.DataFrame) else pd.Series(
            np.asarray(X).reshape(-1)
        )
        return self.encoder_.transform(values.apply(split_genres))

    def get_feature_names_out(self, input_features=None):
        return np.array(
            ["genre_" + str(genre) for genre in self.encoder_.classes_],
            dtype=object
        )


numeric_features = ["release_year", "year_added"]
categorical_features = ["type", "main_country"]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features),
        ("genres", GenreEncoder(), ["listed_in"])
    ],
    remainder="drop",
    verbose_feature_names_out=True
)



# 4. CREATE THE RATING-GROUP TARGET


rating_groups = {
    "TV-Y": "Kids",
    "TV-Y7": "Kids",
    "TV-Y7-FV": "Kids",
    "TV-G": "Kids",
    "G": "Kids",
    "PG": "Family",
    "TV-PG": "Family",
    "PG-13": "Teen",
    "TV-14": "Teen",
    "R": "Adult",
    "TV-MA": "Adult",
    "NC-17": "Adult"
}

model_df = df[df["rating"].isin(rating_groups)].copy()
model_df["target_group"] = model_df["rating"].map(rating_groups)

print("\nRating group distribution:")
print(model_df["target_group"].value_counts())

# The original rating is not an input because it defines the target.
# Duration is also excluded from the feature set.
X = model_df[
    ["release_year", "year_added", "type", "main_country", "listed_in"]
].copy()
y = model_df["target_group"].astype(str)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("\nEligible records:", len(model_df))
print("Excluded unmapped/unrated records:", len(df) - len(model_df))
print("Training records:", len(X_train))
print("Testing records:", len(X_test))



# 5. COMPARE CLASSIFICATION MODELS


cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

models = {
    "Baseline": Pipeline([
        ("preprocessor", preprocessor),
        ("model", DummyClassifier(strategy="most_frequent"))
    ]),
    "Logistic Regression": Pipeline([
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(
            max_iter=5000,
            C=0.1,
            solver="saga",
            tol=1e-3,
            class_weight="balanced",
            random_state=RANDOM_STATE
        ))
    ]),
    "Decision Tree": Pipeline([
        ("preprocessor", preprocessor),
        ("model", DecisionTreeClassifier(
            max_depth=12,
            class_weight="balanced",
            random_state=RANDOM_STATE
        ))
    ]),
    "Random Forest": Pipeline([
        ("preprocessor", preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=250,
            max_depth=20,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=1
        ))
    ]),
    "Extra Trees": Pipeline([
        ("preprocessor", preprocessor),
        ("model", ExtraTreesClassifier(
            n_estimators=250,
            max_depth=None,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=1
        ))
    ])
}

comparison_rows = []
fitted_models = {}

for model_name, model in models.items():
    print(f"\nEvaluating {model_name}...")
    cv_scores = cross_val_score(
        model, X_train, y_train,
        cv=cv, scoring="f1_macro", n_jobs=1
    )
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    fitted_models[model_name] = model

    comparison_rows.append({
        "Model": model_name,
        "CV Macro F1": cv_scores.mean(),
        "Test Accuracy": accuracy_score(y_test, predictions),
        "Test Macro F1": f1_score(
            y_test, predictions,
            labels=GROUP_ORDER, average="macro", zero_division=0
        )
    })



# 6. TUNE EXTRA TREES CLASSIFIER

tuning_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", ExtraTreesClassifier(
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=1
    ))
])

parameter_grid = {
    "model__n_estimators": [150, 250],
    "model__max_depth": [20, None],
    "model__min_samples_leaf": [2, 4]
}

search = GridSearchCV(
    estimator=tuning_pipeline,
    param_grid=parameter_grid,
    scoring="f1_macro",
    cv=cv,
    n_jobs=1,
    refit=True
)

print("\nTuning Extra Trees Classifier...")
search.fit(X_train, y_train)

tuned_model = search.best_estimator_
tuned_predictions = tuned_model.predict(X_test)
fitted_models["Tuned Extra Trees"] = tuned_model

comparison_rows.append({
    "Model": "Tuned Extra Trees",
    "CV Macro F1": search.best_score_,
    "Test Accuracy": accuracy_score(y_test, tuned_predictions),
    "Test Macro F1": f1_score(
        y_test, tuned_predictions,
        labels=GROUP_ORDER, average="macro", zero_division=0
    )
})

comparison_df = (
    pd.DataFrame(comparison_rows)
    .sort_values("CV Macro F1", ascending=False)
    .reset_index(drop=True)
    .round(3)
)

print("\nModel comparison:")
print(comparison_df.to_string(index=False))
print("\nTuned Extra Trees parameters:")
print(search.best_params_)

comparison_df.to_csv("results/task6_model_comparison.csv", index=False)

# Select by cross-validation Macro F1. The test set is held back for evaluation.
selected_name = comparison_df.iloc[0]["Model"]
best_model = fitted_models[selected_name]
best_predictions = best_model.predict(X_test)

selected_cv_f1 = comparison_df.iloc[0]["CV Macro F1"]
selected_accuracy = accuracy_score(y_test, best_predictions)
selected_macro_f1 = f1_score(
    y_test, best_predictions,
    labels=GROUP_ORDER, average="macro", zero_division=0
)

print("\nSelected model:", selected_name)
print("Selected model CV Macro F1:", selected_cv_f1)
print("Selected model test accuracy:", round(selected_accuracy, 3))
print("Selected model test Macro F1:", round(selected_macro_f1, 3))

with open("results/task6_model_summary.txt", "w", encoding="utf-8") as file:
    file.write(f"Selected model: {selected_name}\n")
    file.write(f"Best Extra Trees parameters: {search.best_params_}\n")
    file.write(f"Selected model CV Macro F1: {selected_cv_f1:.4f}\n")
    file.write(f"Selected model test accuracy: {selected_accuracy:.4f}\n")
    file.write(f"Selected model test Macro F1: {selected_macro_f1:.4f}\n")



# 7. EVALUATE SELECTED MODEL


report = classification_report(
    y_test, best_predictions,
    labels=GROUP_ORDER, zero_division=0
)
print("\nSelected model classification report:")
print(report)

with open("results/task6_classification_report.txt", "w", encoding="utf-8") as file:
    file.write(f"Selected model: {selected_name}\n\n")
    file.write(report)

fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_predictions(
    y_test, best_predictions,
    labels=GROUP_ORDER,
    display_labels=GROUP_ORDER,
    cmap="Blues",
    colorbar=False,
    ax=ax
)
ax.set_title(f"Rating Group Prediction: {selected_name}")
plt.tight_layout()
plt.savefig("figures/task6_confusion_matrix.png", dpi=150)
plt.close()

# 8. FEATURE IMPORTANCE

fitted_preprocessor = best_model.named_steps["preprocessor"]
fitted_estimator = best_model.named_steps["model"]
feature_names = fitted_preprocessor.get_feature_names_out()

importance = None
if hasattr(fitted_estimator, "feature_importances_"):
    importance = pd.Series(
        fitted_estimator.feature_importances_, index=feature_names
    )
elif hasattr(fitted_estimator, "coef_"):
    coefficients = np.asarray(fitted_estimator.coef_)
    importance = pd.Series(
        np.abs(coefficients).mean(axis=0), index=feature_names
    )

if importance is not None:
    importance = importance.sort_values(ascending=False)
    top_importance = importance.head(15)
    print("\nTop predictive features:")
    print(top_importance.to_string())

    top_importance.sort_values().plot(kind="barh", figsize=(9, 6))
    plt.title(f"Top 15 Features: {selected_name}")
    plt.xlabel("Feature importance / mean absolute coefficient")
    plt.tight_layout()
    plt.savefig("figures/task6_feature_importance.png", dpi=150)
    plt.close()
    importance.to_csv("results/task6_feature_importance.csv", header=["Importance"])
else:
    print("\nSelected model does not provide feature importance or coefficients.")



# 9. UNSUPERVISED LEARNING: K-MEANS CLUSTERING


print("\nRunning K-Means clustering...")

cluster_input = df[
    ["release_year", "year_added", "type", "main_country", "listed_in"]
].copy()
cluster_features = preprocessor.fit_transform(cluster_input)
cluster_features = np.asarray(cluster_features, dtype=np.float64)

kmeans = KMeans(n_clusters=4, random_state=RANDOM_STATE, n_init=10)
cluster_labels = kmeans.fit_predict(cluster_features)

cluster_df = df.copy()
cluster_df["cluster"] = cluster_labels
cluster_df[
    ["title", "type", "release_year", "rating", "main_country", "listed_in", "cluster"]
].to_csv("results/task6_clustered_titles.csv", index=False)

print("\nCluster sizes:")
print(cluster_df["cluster"].value_counts().sort_index())

cluster_summary = cluster_df.groupby("cluster").agg(
    title_count=("title", "count"),
    average_release_year=("release_year", "mean"),
    most_common_type=("type", lambda values: values.mode().iloc[0] if not values.mode().empty else "Unknown"),
    most_common_country=("main_country", lambda values: values.mode().iloc[0] if not values.mode().empty else "Unknown"),
    most_common_rating=("rating", lambda values: values.mode().iloc[0] if not values.mode().empty else "Unknown")
).round(1)

print("\nCluster summary:")
print(cluster_summary.to_string())
cluster_summary.to_csv("results/task6_cluster_summary.csv")

cluster_df["cluster"].value_counts().sort_index().plot(kind="bar", figsize=(7, 4))
plt.title("Netflix Content Clusters")
plt.xlabel("Cluster")
plt.ylabel("Number of titles")
plt.tight_layout()
plt.savefig("figures/task6_cluster_distribution.png", dpi=150)
plt.close()

pca = PCA(n_components=2, random_state=RANDOM_STATE)
pca_values = pca.fit_transform(cluster_features)

plt.figure(figsize=(9, 6))
for cluster_id in sorted(np.unique(cluster_labels)):
    mask = cluster_labels == cluster_id
    plt.scatter(
        pca_values[mask, 0], pca_values[mask, 1],
        label=f"Cluster {cluster_id}", alpha=0.6, s=18
    )
plt.title("Netflix Content Clusters (PCA Projection)")
plt.xlabel("Principal component 1")
plt.ylabel("Principal component 2")
plt.legend()
plt.tight_layout()
plt.savefig("figures/task6_cluster_visualization.png", dpi=150)
plt.close()



# 10. AUTOMATED BUSINESS INSIGHTS



insights = [
    f"Dataset contains {len(df):,} unique title/type/release-year records."
]

type_pct = df["type"].value_counts(normalize=True).mul(100).round(1)
if not type_pct.empty:
    insights.append(
        f"{type_pct.index[0]} is the most common content type "
        f"({type_pct.iloc[0]}% of records)."
    )

if not top_genres.empty:
    insights.append(
        f"The most frequently listed genre is '{top_genres.index[0]}' "
        f"({top_genres.iloc[0]} occurrences)."
    )

if not top_countries.empty:
    insights.append(
        f"The most frequently listed main country is '{top_countries.index[0]}' "
        f"({top_countries.iloc[0]} titles)."
    )

if not release_counts.empty:
    insights.append(
        f"Among release years from 1990 to 2025, {int(release_counts.idxmax())} "
        f"has the highest title count in this dataset ({release_counts.max()} titles)."
    )

if not added_counts.empty:
    insights.append(
        f"{int(added_counts.idxmax())} is the year with the most recorded additions "
        f"({added_counts.max()} titles)."
    )

insights.append(
    "Rating prediction estimates the dataset's rating category; it does not "
    "measure audience popularity or commercial success."
)
insights.append(
    "K-Means clusters are exploratory groups based on metadata, not verified "
    "success categories."
)

insights_text = "\n".join(
    f"{index + 1}. {insight}" for index, insight in enumerate(insights)
)
print("\n" + "=" * 65)
print("AUTOMATED CONTENT INSIGHTS")
print("=" * 65)
print(insights_text)

with open("results/task6_automated_insights.txt", "w", encoding="utf-8") as file:
    file.write(insights_text)



# 11. FINAL OUTPUT SUMMARY


print("\n" + "=" * 65)
print("TASK 6 COMPLETED - REVIEW METRICS BEFORE SUBMISSION")
print("=" * 65)

print("\nMain outputs:")
for path in [
    "results/task6_cleaned_dataset.csv",
    "results/task6_model_comparison.csv",
    "results/task6_classification_report.txt",
    "results/task6_model_summary.txt",
    "results/task6_feature_importance.csv",
    "results/task6_clustered_titles.csv",
    "results/task6_cluster_summary.csv",
    "results/task6_automated_insights.txt"
]:
    print(path)

print("\nFigures:")
for path in [
    "figures/task6_content_type.png",
    "figures/task6_top_genres.png",
    "figures/task6_top_countries.png",
    "figures/task6_release_year_trend.png",
    "figures/task6_year_added.png",
    "figures/task6_confusion_matrix.png",
    "figures/task6_feature_importance.png",
    "figures/task6_cluster_distribution.png",
    "figures/task6_cluster_visualization.png"
]:
    print(path)

print("\nReview all metrics and insights before submission.")
print("Do not describe metadata patterns as actual audience success.")


NETFLIX CONTENT ANALYTICS & INSIGHTS ENGINE
Main model candidate: Extra Trees Classifier

Dataset shape: (8787, 13)

Content type distribution:
type
Movie      6124
TV Show    2663
Name: count, dtype: Int64

Missing values:
show_id                0
type                   0
title                  0
director               0
country              287
date_added             0
release_year           0
rating                 0
duration               0
listed_in              0
date_added_parsed      0
year_added             0
main_country           0
dtype: int64

Rating group distribution:
target_group
Adult     4006
Teen      2645
Family    1148
Kids       906
Name: count, dtype: int64

Eligible records: 8705
Excluded unmapped/unrated records: 82
Training records: 6964
Testing records: 1741

Evaluating Baseline...

Evaluating Logistic Regression...

Evaluating Decision Tree...

Evaluating Random Forest...

Evaluating Extra Trees...

Tuning Extra Trees Classifier...

Model comparison:
       